# Bag-of-Words (Bolsa de Palabras - BoW) para detección de Spam en español

## Introducción

Este notebook demuestra el modelo **Bag-of-Words (BoW)** (Bolsa de Palabras), una técnica fundamental en el Procesamiento de Lenguaje Natural (PLN) para representar datos de texto numéricamente.

Vamos a:
1.  Explicar el concepto de Bag-of-Words.
2.  Cargar un conjunto de datos en español desde Hugging Face (`tanaos/synthetic-spam-detection-dataset-spanish`).
3.  Aplicar la técnica BoW utilizando `CountVectorizer` adaptado al español.
4.  Entrenar un modelo de clasificación simple (Naive Bayes Multinomial).
5.  Evaluar el modelo y hacer predicciones sobre nuevos mensajes en español.

## ¿Qué es Bag-of-Words?

El modelo Bag-of-Words (Bolsa de Palabras) es una forma de representar datos de texto al modelarlos con algoritmos de machine learning. Simplifica el texto enfocándose *únicamente* en la **ocurrencia (frecuencia) de las palabras** dentro de un documento, ignorando por completo **la gramática y el orden de las palabras**.

**Idea principal:**

Se tiene una colección de documentos. El modelo BoW trata cada documento como una "bolsa" que contiene sus palabras.

1.  **Tokenización:** el texto se divide en palabras individuales (tokens).
2.  **Construcción del vocabulario:** se crea un vocabulario con todas las palabras únicas presentes en el conjunto de entrenamiento.
3.  **Vectorización:** cada documento se convierte en un vector numérico. La longitud de este vector es igual al tamaño del vocabulario. El valor en cada posición representa cuántas veces aparece esa palabra específica en el documento.

**Punto Clave:** BoW convierte documentos de texto en vectores numéricos de tamaño fijo, haciéndolos adecuados para los algoritmos. Sin embargo, pierde información sobre el orden y el contexto.

## Configuración e instalación

Instalamos e importamos las bibliotecas necesarias, incluyendo `datasets` para descargar los datos y `nltk` para procesar el texto en español.

In [ ]:
#!pip install datasets nltk -q

In [ ]:
import pandas as pd
import numpy as np
import nltk
from datasets import load_dataset
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Descargar las palabras vacías (stop words) en español
nltk.download('stopwords', quiet=True)

## Cargar datos

Usaremos el dataset `tanaos/synthetic-spam-detection-dataset-spanish` desde Hugging Face. Contiene mensajes sintéticos de SMS etiquetados con `0` (no spam) y `1` (spam).

In [2]:
from datasets import load_dataset
# Cargar el conjunto de datos desde Hugging Face
dataset = load_dataset("tanaos/synthetic-spam-detection-dataset-spanish")

# Convertir la partición de entrenamiento (train) a un DataFrame de Pandas
df = dataset['train'].to_pandas()

print("Primeras filas del dataset:")
print(df.head())
print("\nInformación del dataset:")
df.info()

C:\Users\laris\Downloads\InteligenciaArtificial\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\laris\Downloads\InteligenciaArtificial\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\laris\.cache\huggingface\hub\datasets--tanaos--synthetic-spam-detection-dataset-spanish. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode

Primeras filas del dataset:
                                                text  labels
0  Hola, ¿cómo estás? Espero que tengas un buen día.       0
1  Acabo de terminar el informe que me asignaron,...       0
2  ¿Te gustaría comprar entradas para el conciert...       1
3  Recordatorio: la reunión del equipo será mañan...       0
4  Oferta especial: suscríbete ahora y recibe un ...       1

Información del dataset:
<class 'pandas.DataFrame'>
RangeIndex: 15016 entries, 0 to 15015
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    15016 non-null  str  
 1   labels  15016 non-null  int64
dtypes: int64(1), str(1)
memory usage: 1.4 MB


## Preparar datos

En este dataset, las etiquetas numéricas ya vienen preparadas en la columna `labels` y el texto en la columna `text`. Solo necesitamos separar características (X) y objetivo (y), y dividir los datos en entrenamiento y prueba.

In [4]:
# Definir características (X) y objetivo (y)

from sklearn.model_selection import train_test_split

X = df['text']
y = df['labels']

# Dividir los datos en conjuntos de entrenamiento y prueba (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Tamaño del conjunto de entrenamiento: {X_train.shape[0]} mensajes")
print(f"Tamaño del conjunto de prueba: {X_test.shape[0]} mensajes")

print("-----------------------")
print("Primeras filas de X:")
print(X.head())
print("-----------------------")
print("Primeras filas de y:")
print(y.head())

Tamaño del conjunto de entrenamiento: 12012 mensajes
Tamaño del conjunto de prueba: 3004 mensajes
-----------------------
Primeras filas de X:
0    Hola, ¿cómo estás? Espero que tengas un buen día.
1    Acabo de terminar el informe que me asignaron,...
2    ¿Te gustaría comprar entradas para el conciert...
3    Recordatorio: la reunión del equipo será mañan...
4    Oferta especial: suscríbete ahora y recibe un ...
Name: text, dtype: str
-----------------------
Primeras filas de y:
0    0
1    0
2    1
3    0
4    1
Name: labels, dtype: int64


## Aplicando Bag-of-Words

Ahora convertiremos los textos en vectores numéricos. Para mejorar el modelo en español, le pasaremos una lista de **stop words** (palabras vacías como "el", "la", "de", "y") generadas por la librería `nltk`. Estas palabras son tan comunes que no aportan valor para detectar spam.

**Pasos Clave:**
1.  **Inicializar `CountVectorizer`**: usar la lista de stop words en español.
2.  **Ajustar (Fit)**: solo con los datos de entrenamiento (`X_train`) para construir el vocabulario.
3.  **Transformar**: convertir `X_train` y `X_test` en matrices numéricas dispersas.

In [5]:
# Obtener palabras vacías en español
stop_words_es = stopwords.words('spanish')

# 1. Inicializar CountVectorizer excluyendo las palabras vacías en español
vectorizer = CountVectorizer(stop_words=stop_words_es)

# 2. Ajustar el vectorizador en los datos de entrenamiento para construir el vocabulario
vectorizer.fit(X_train)

print(f"Tamaño del vocabulario: {len(vectorizer.vocabulary_)}")

# 3. Transformar los datos de entrenamiento y prueba en vectores BoW (matriz dispersa)
X_train_bow = vectorizer.transform(X_train)
X_test_bow = vectorizer.transform(X_test)

print(f"Tamaño de la matriz BoW de entrenamiento: {X_train_bow.shape}")
print(f"Tamaño de la matriz BoW de prueba: {X_test_bow.shape}")

print("\nVector BoW para el primer mensaje de entrenamiento (disperso):")
print(X_train_bow[0])
print("\nVector BoW para el primer mensaje (denso, mostrando las primeras 50 características):")
print(X_train_bow[0].toarray()[0, :50])

NameError: name 'stopwords' is not defined

La salida `(0, word_index) count` de la matriz dispersa significa: en el primer documento (índice 0), la palabra correspondiente a `word_index` en el vocabulario apareció `count` veces.

## Entrenando un modelo simple

Usaremos **Naive Bayes Multinomial (MNB)**, que suele ser un excelente modelo de referencia para clasificación de texto.

In [ ]:
# Inicializar el clasificador Naive Bayes Multinomial
model = MultinomialNB()

# Entrenar el modelo utilizando la matriz BoW
model.fit(X_train_bow, y_train)

print("Entrenamiento del modelo completado.")

## Haciendo predicciones

Evaluamos sobre el **conjunto de prueba** (`X_test_bow`).

In [ ]:
# Predecir etiquetas para el conjunto de prueba
y_pred = model.predict(X_test_bow)

## Evaluando el modelo

Verificamos la calidad del modelo utilizando métricas de clasificación.

In [ ]:
# Calcular exactitud (accuracy)
accuracy = accuracy_score(y_test, y_pred)
print(f"Exactitud (Accuracy): {accuracy:.4f}")

# Mostrar la matriz de confusión
print("\nMatriz de Confusión:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

In [ ]:
#!pip install matplotlib
#!pip install seaborn

In [6]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar el tamaño de la figura
plt.figure(figsize=(7, 5))

# Crear un mapa de calor (heatmap) estilizado con Seaborn
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Spam (0)', 'Spam (1)'], 
            yticklabels=['No Spam (0)', 'Spam (1)'], 
            linewidths=.5, cbar=False, annot_kws={"size": 14})

plt.title('Matriz de confusión', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Predicción del modelo', fontsize=12, labelpad=10)
plt.ylabel('Valor real', fontsize=12, labelpad=10)

# Ajustar el diseño y mostrar el gráfico
plt.tight_layout()
plt.show()

NameError: name 'cm' is not defined

<Figure size 700x500 with 0 Axes>

*   **Matriz de confusión:**
    *   Arriba-izquierda: Verdaderos Negativos (Predijo correctamente 'No Spam')
    *   Abajo-derecha: Verdaderos Positivos (Predijo correctamente 'Spam')
    *   Arriba-derecha: Falsos Positivos (Predijo 'Spam' pero no lo era)
    *   Abajo-izquierda: Falsos Negativos (Predijo 'No Spam' pero era Spam)

In [ ]:
# Mostrar el reporte de clasificación
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred, target_names=['No Spam (0)', 'Spam (1)']))

*   **Puntuación F1 (F1-Score):** Media de Precisión y Exhaustividad. Un valor cercano a 1 indica buen rendimiento.

## Ejemplo de predicción con nuevos mensajes reales

Vamos a probar nuestro modelo con frases creadas por nosotros. Es vital usar el **mismo `vectorizer`** para que las nuevas palabras coincidan con el vocabulario entrenado.

In [ ]:
# Nuevos mensajes de texto en español a clasificar
new_messages = [
    "Hola, ¿cómo estás hoy? ¿Nos vemos luego para almorzar?",
    "¡Felicidades! Tu línea ha sido seleccionada para ganar un iPhone 15. Haz clic en este enlace confidencial para reclamar tu premio.",
    "No olvides traer el informe a la reunión de las 10 am.",
    "PRÉSTAMO INMEDIATO PRE-APROBADO. Sin aval. Llama ya al 0800-DINERO-RAPIDO.",
    "Ganate un iPhone hoy. Llamanos al 11-3522-6792",
    "Te contactamos del servicio de vacunación contra la gripe. Envianos el número que aparece en pantalla para agendar tu vacuna.",
    "Doná $ 3.000 para los soldados de la guerra de Ukrania",
    "Hola Charlie: te puedo llamar en 15?",
    "Hoy es el cumple de Emilio. Junto plata para el regalo en marian.felice.mp"
]

# Transformar los nuevos mensajes usando el vectorizador *ajustado*
new_messages_bow = vectorizer.transform(new_messages)

# Predecir
new_predictions = model.predict(new_messages_bow)

# Mapear predicciones a texto
label_map = {0: 'No Spam', 1: '¡SPAM!'}
predicted_labels = [label_map[pred] for pred in new_predictions]

# Resultados
for msg, label in zip(new_messages, predicted_labels):
    print(f'Mensaje: "{msg}" \n--> Predicción: {label}\n')

## Conclusión

Este notebook demostró el uso de Bag-of-Words directamente en español utilizando un dataset extraído de Hugging Face y palabras vacías (`stop words`) de `nltk`.

*   Al eliminar las palabras vacías españolas, reducimos ruido en nuestros vectores.
*   El modelo `MultinomialNB` funciona extremadamente rápido e identifica de forma certera patrones repetitivos del spam.
*   A pesar de las limitaciones de BoW (como no entender ironía o contexto), sigue siendo una técnica súper eficiente y difícil de superar en rendimiento vs. coste computacional para este tipo de tareas.